# Calliope Notes

Here I will provide detailed explanations for different things that I study in Calliope and that later will be used for creating visualizations.

Just for illustration purposes I will use the Urban_Scale example from the documentation.

In [37]:
import calliope

# We increase logging verbosity
calliope.set_log_verbosity('INFO', include_solver_output=False)

In [38]:
model = calliope.Model("urban_scale/model.yaml")

[2025-09-22 09:25:32] INFO     Model: initialising
[2025-09-22 09:25:32] INFO     Model: preprocessing stage 1 (model_run)
[2025-09-22 09:25:32] INFO     Model: preprocessing stage 2 (model_data)
[2025-09-22 09:25:32] INFO     Model: preprocessing complete


## Model.inputs

There are generally 2 ways of parsing the input data. One way is obviously parsing of initial `.yaml` files and writing all the logic ourselves. While it seems easy at first, it requires full understanding of inner-mechanisms of Calliope, cause a lot of variables in the `.yaml` files often overwrite each other. This is why I decided to have a look into the 2nd approach - looking directly into `Model.inputs`. This is the data retrieved from the same `.yaml` files, but already preprocessed and ready to be used in the simulation.

While one can easily find the documentation for different fields of `.yaml` files, there is almost no easy-readable information on the fields in `Model.inputs`. This is why I will write a small documentation for the most crucial variables, so that one could directly use those, without need to parse `.yaml` files.


### Dimensions

The data in inputs is represented in the form of **xarray.Dataset** - some kind of dict with a lot of multi-dimensional arrays inside. The arrays must follow some coordinate system, described when creating the Dataset using `coords` parameter.

So, for `Model.inputs`, the names of Dimensions and their Coordinates are:

- carrier_tiers - TBC
- `carriers` - list of possible energy carriers. For example, it can be *electricity*, *gas*, *heat*.
- `costs` - each technology has some costs, most often those are *monetary*(how much $ it costs) or *environmental*(can be represented in the amount of emissions of co2). Generally means something bad, something, that our model will try to minimize.
- `coordinates` - list of names of dimensions on the map (when specifying locations). TBC
- `loc_carriers` - list of combinations of all possible "`location_name`::`carrier`" such that the exact `carrier` is present in exactly this `location_name`.
- `loc_tech_carriers_conversion_plus` - list of combinations of "`location_name`::`conv_plus_tech_name`::`carrier`" such that there is a `conv_plus_tech_name` located at `location_name` and somehow processes `carrier`.
- `loc_techs` - a list of combinations of "`location_name`::`tech`" such that this exact `tech` is present in that exact `location_name`.
- `loc_techs_area` - a subset of `loc_techs` such that technologies in this list are somehow affected or restricted by area (for example, PV tech is usually restricted by available area for them).
- `loc_techs_conversion` - a subset of `loc_techs` such that technologies in this list are of `conversion` type (excluding `conversion_plus`).
- `loc_techs_conversion_plus` - a subset of `loc_techs` such that technologies in this list are of `conversion_plus` type.
- `loc_techs_export` - a subset of `loc_techs` such that technologies in this list are allowed to export some carriers outside the system. This can be defined using the `export_carrier` property of tech in `techs.yaml` If some tech is not included in this list it still means it is able to share the carrier with other locations as long as there is appropriate transmission infrastructure; the only restriction is to export carrier outside the system(for example, in the national grid).
- `loc_techs_finite_resource` - a subset of `loc_techs` such that the technologies there have finite resource constraints. This usually includes all the demands in the system and supply technologies that are somehow restricted in the amount of producing carrier. **ToBeChecked - something else might be also included!**
- `loc_techs_investment_cost` - a subset of `loc_techs` such that technologies in this list require some kind of investment before the tech can be used. This category includes only "buying the equipment" cost, but not operational. This cost does not depend on how much one uses this tech later.
- `loc_techs_non_conversion` - a subset of `loc_techs` such that technologies in this list do not convert one type of carrier into another.
- `loc_techs_om_cost` - a subset of `loc_techs` such that technologies in this list require some kind of investment during usage. This can be, for example, price one needs to pay per unit of carriage.
- `loc_techs_supply_plus` - a subset of `loc_techs` such that technologies in this list are of `supply_plus` type.
- `loc_techs_transmission` - a subset of `loc_techs` such that technologies in this list are of `transmission` type. The transmission is in the format "source_location_name::transmission_tech_name:destination_location_name".
- `locs` - an array of all locations.
- `techs` - an array of all technologies.
- `timesteps` - an array of all timesteps.

### Data Variables:

Exactly the arrays with described dimensions from above. So, for example, data variable `available_area ('locs',)` means that there is an array `available_area` with one just dimension `'locs'`, and basically represents the amount of available area in every location. So, if one wants to find out the amount of available area in location "X2", one must first check the index of this location at `model.inputs.locs`, and then extract the value from the `available_area` array using this index.


- `available_area` ('locs',) - array of available area in every location. It is specified in the `locations.yaml`.
- `carrier_ratios` ('carrier_tiers', 'loc_tech_carriers_conversion_plus') - **ToDo**
- `colors` ('techs',) - specifies the color for each location for each tech. It is specified in the `techs.yaml`.
- `cost_depreciation_rate` ('costs', 'loc_techs_investment_cost') - in order to compare the invest costs for each tech with varying lifetime and built costs, deprecation_rate for each tech is calculated. The idea behind is that instead of paying all built costs upfront one can borrow money from the bank with the specified interest rate for the period of tech lifetime; and when comparing the costs one can compare yearly payments to the bank that are calculated using deprecation_rate (`yearly_payment = upfront_pay * deprecation_rate`). So this 2D array contains deprecation rates for each technology for every type of costs(that can be in $, co2, etc.)
- `cost_energy_cap` ('costs', 'loc_techs_investment_cost') -
- `cost_export` ('costs', 'loc_techs_om_cost', 'timesteps') -
- `cost_om_annual` ('costs', 'loc_techs_investment_cost') -
- `cost_om_con` ('costs', 'loc_techs_om_cost') -
- `cost_om_prod` ('costs', 'loc_techs_om_cost') -
- `distance` ('loc_techs_transmission',) -
- `energy_cap_max` ('loc_techs',) -
- `energy_con` ('loc_techs',) -
- `energy_eff` ('loc_techs',) -
- `energy_prod` ('loc_techs',) -
- `export_carrier` ('loc_techs_export',) -
- `force_resource` ('loc_techs_finite_resource',) -
- `inheritance` ('techs',) -
- `lifetime` ('loc_techs',) -
- `loc_coordinates` ('coordinates', 'locs') -
- `lookup_loc_carriers` ('loc_carriers',) -
- `lookup_loc_techs` ('loc_techs_non_conversion',) -
- `lookup_loc_techs_area` ('locs',) -
- `lookup_loc_techs_conversion` ('carrier_tiers', 'loc_techs_conversion') -
- `lookup_loc_techs_conversion_plus` ('carrier_tiers', 'loc_techs_conversion_plus') -
- `lookup_loc_techs_export` ('loc_techs_export',) -
- `lookup_primary_loc_tech_carriers_in` ('loc_techs_conversion_plus',) -
- `lookup_primary_loc_tech_carriers_out` ('loc_techs_conversion_plus',) -
- `lookup_remotes` ('loc_techs_transmission',) -
- `max_demand_timesteps` ('carriers',) -
- `names` ('techs',) -
- `parasitic_eff` ('loc_techs_supply_plus',) -
- `resource` ('loc_techs_finite_resource', 'timesteps') -
- `resource_area_max` ('loc_techs_area',) -
- `resource_area_per_energy_cap` ('loc_techs_area',) -
- `resource_eff` ('loc_techs_finite_resource',) -
- `resource_unit` ('loc_techs_finite_resource',) -
- `timestep_resolution` ('timesteps',) -
- `timestep_weights` ('timesteps',) -







In [43]:
model.inputs.cost_depreciation_rate

<xarray.DataArray 'cost_depreciation_rate' (costs: 1,
                                            loc_techs_investment_cost: 20)>
array([[0.11016807, 0.11016807, 0.11016807, 0.11016807, 0.11016807,
        0.11016807, 0.11016807, 0.11016807, 0.11016807, 0.11016807,
        0.11016807, 0.11016807, 0.11016807, 0.11016807, 0.11016807,
        0.11016807, 0.11016807, 0.11016807, 0.11016807, 0.11016807]])
Coordinates:
  * costs                      (costs) object 'monetary'
  * loc_techs_investment_cost  (loc_techs_investment_cost) object 'X2::boiler...
Attributes:
    is_result:  0

In [40]:
for data_var in sorted(model.inputs):
    name = data_var
    dimensions = model.inputs[name].dims

    print(f'- `{name}` {dimensions} - ')

- `available_area` ('locs',) - 
- `carrier_ratios` ('carrier_tiers', 'loc_tech_carriers_conversion_plus') - 
- `colors` ('techs',) - 
- `cost_depreciation_rate` ('costs', 'loc_techs_investment_cost') - 
- `cost_energy_cap` ('costs', 'loc_techs_investment_cost') - 
- `cost_export` ('costs', 'loc_techs_om_cost', 'timesteps') - 
- `cost_om_annual` ('costs', 'loc_techs_investment_cost') - 
- `cost_om_con` ('costs', 'loc_techs_om_cost') - 
- `cost_om_prod` ('costs', 'loc_techs_om_cost') - 
- `distance` ('loc_techs_transmission',) - 
- `energy_cap_max` ('loc_techs',) - 
- `energy_con` ('loc_techs',) - 
- `energy_eff` ('loc_techs',) - 
- `energy_prod` ('loc_techs',) - 
- `export_carrier` ('loc_techs_export',) - 
- `force_resource` ('loc_techs_finite_resource',) - 
- `inheritance` ('techs',) - 
- `lifetime` ('loc_techs',) - 
- `loc_coordinates` ('coordinates', 'locs') - 
- `lookup_loc_carriers` ('loc_carriers',) - 
- `lookup_loc_techs` ('loc_techs_non_conversion',) - 
- `lookup_loc_techs_area`